# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import pandas as pd
import numpy as np

# Load the anonymized FlyRank dataset directly from the repository
data_url = "https://raw.githubusercontent.com/SriKeerthi16-b/Machine-Learning/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

required_columns = [
    "content_id",
    "days_since_last_update",
    "impressions_90d",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Rows:", len(df))

print("\nSignal summary:")
print(
    df[
        ["days_since_last_update", "impressions_90d"]
    ].describe()
)

# Define the two signals
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500

# Create reason codes
df["reason_code"] = np.select(
    [
        stale & visible,
        stale & ~visible,
        ~stale & visible,
    ],
    [
        "stale_and_visible",
        "stale_not_visible",
        "fresh_and_visible",
    ],
    default="fresh_not_visible",
)

# Create action
df["suggested_action"] = np.where(
    df["reason_code"] == "stale_and_visible",
    "refresh_review",
    "monitor",
)

print("\nReason-code counts:")
print(df["reason_code"].value_counts())

Rows: 30000

Signal summary:
       days_since_last_update  impressions_90d
count            30000.000000     30000.000000
mean                46.098300      5200.366300
std                 42.078709     16838.019547
min                  1.000000         1.000000
25%                 20.000000        81.000000
50%                 20.000000       731.000000
75%                104.000000      3615.250000
max                373.000000    517715.000000

Reason-code counts:
reason_code
fresh_and_visible    16709
fresh_not_visible    13117
stale_not_visible      157
stale_and_visible       17
Name: count, dtype: int64


### Rule

I will prioritize a page for refresh review when it has not been updated for at least 180 days and still has at least 500 search impressions in the trailing 90-day window.

The rule uses two observed signals:
- `days_since_last_update >= 180` for staleness.
- `impressions_90d >= 500` for meaningful search visibility.

### Reason codes

- `stale_and_visible`: the page is at least 180 days since its last update and has at least 500 impressions.
- `stale_not_visible`: the page is stale but has fewer than 500 impressions.
- `fresh_and_visible`: the page has at least 500 impressions but is not yet stale.
- `fresh_not_visible`: the page is neither stale nor meaningfully visible.

### Action

Pages with `stale_and_visible` receive the `refresh_review` action. Other pages receive `monitor`.

This is a transparent, rule-based decision-support baseline, not a prediction.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Calculate relative scores for the two signals

df["freshness_risk_score"] = (
    df["days_since_last_update"].rank(pct=True)
)

df["visibility_score"] = (
    df["impressions_90d"].rank(pct=True)
)

# Apply the rule
rule_match = df["reason_code"] == "stale_and_visible"

# Calculate baseline score
df["baseline_action_score"] = np.where(
    rule_match,
    0.5 * df["freshness_risk_score"]
    + 0.5 * df["visibility_score"],
    0.0
)

# Rank all pages
df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Create output folder
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Create ranked queue
output_columns = [
    "content_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "suggested_action",
    "days_since_last_update",
    "impressions_90d",
]

queue = (
    df[output_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

# Write CSV
queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("CSV written to:", output_path)

print("\nTop 20:")
display(queue.head(20))

Rows ranked: 30000
CSV written to: work/outputs/baseline_action_score.csv

Top 20:


,content_id,baseline_rank,baseline_action_score,reason_code,suggested_action,days_since_last_update,impressions_90d
0,content_cf56e2e2e282,1,0.991317,stale_and_visible,refresh_review,194,61678
1,content_7368877ea310,2,0.991117,stale_and_visible,refresh_review,194,59472
2,content_1bfaa38ff26c,3,0.975817,stale_and_visible,refresh_review,194,25715
3,content_0a91db491d14,4,0.951967,stale_and_visible,refresh_review,193,13299
4,content_5feee3994adb,5,0.924433,stale_and_visible,refresh_review,194,7812
5,content_c2d929d83eaa,6,0.922350,stale_and_visible,refresh_review,193,7558
6,content_b16bd7307b39,7,0.889992,stale_and_visible,refresh_review,194,4590
7,content_fe16a55cd13d,8,0.889442,stale_and_visible,refresh_review,194,4556
8,content_ecb6215e79fd,9,0.887242,stale_and_visible,refresh_review,194,4429
9,content_928af3e22c80,10,0.814308,stale_and_visible,refresh_review,193,1697


### Scoring approach

I rank pages using a transparent baseline score.

The score combines relative staleness and search visibility. Pages receive a positive score when they satisfy the `stale_and_visible` rule. Higher staleness and higher visibility increase the score.

Pages that do not satisfy the rule receive a score of zero and remain lower-priority monitoring items.

The ranked queue is written to `work/outputs/baseline_action_score.csv`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Select the top 20 ranked pages
top20 = queue.head(20).copy()

# Add confidence notes
top20["confidence_note"] = np.where(
    top20["reason_code"] == "stale_and_visible",
    "Moderate: both rule conditions are observed, but this is a heuristic.",
    "Low: the page does not meet the full refresh-review rule."
)

# Add what could make the recommendation wrong
top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "stale_and_visible",
        top20["reason_code"] == "stale_not_visible",
        top20["reason_code"] == "fresh_and_visible",
    ],
    [
        "The content may still be accurate despite being old.",
        "Low visibility may mean refresh effort has limited measurable value.",
        "The page is visible but has been updated recently.",
    ],
    default="The page has neither strong staleness nor visibility under this rule."
)

review_columns = [
    "baseline_rank",
    "content_id",
    "suggested_action",
    "reason_code",
    "baseline_action_score",
    "days_since_last_update",
    "impressions_90d",
    "confidence_note",
    "what_would_make_it_wrong",
]

top20_review = top20[review_columns]

display(top20_review)

,baseline_rank,content_id,suggested_action,reason_code,baseline_action_score,days_since_last_update,impressions_90d,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,refresh_review,stale_and_visible,0.991317,194,61678,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
1,2,content_7368877ea310,refresh_review,stale_and_visible,0.991117,194,59472,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
2,3,content_1bfaa38ff26c,refresh_review,stale_and_visible,0.975817,194,25715,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
3,4,content_0a91db491d14,refresh_review,stale_and_visible,0.951967,193,13299,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
4,5,content_5feee3994adb,refresh_review,stale_and_visible,0.924433,194,7812,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
5,6,content_c2d929d83eaa,refresh_review,stale_and_visible,0.922350,193,7558,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
6,7,content_b16bd7307b39,refresh_review,stale_and_visible,0.889992,194,4590,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
7,8,content_fe16a55cd13d,refresh_review,stale_and_visible,0.889442,194,4556,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
8,9,content_ecb6215e79fd,refresh_review,stale_and_visible,0.887242,194,4429,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...
9,10,content_928af3e22c80,refresh_review,stale_and_visible,0.814308,193,1697,"Moderate: both rule conditions are observed, b...",The content may still be accurate despite bein...


### Top-20 review

The top 20 rows are reviewed using the baseline action, reason code, confidence note, and a condition that could make the recommendation wrong.

The confidence note is a qualitative description of how directly the row matches the rule. It is not a statistical confidence estimate.

Each recommendation is intended as decision-support for human review rather than an automatic content change.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check which features were used by the baseline score

score_features = [
    "days_since_last_update",
    "impressions_90d",
]

# Fields that should not be used as scoring inputs
forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

used_forbidden = [
    col for col in score_features
    if col in forbidden_fields
]

print("Features used by the score:")
print(score_features)

print("\nForbidden outcome/future fields used:")
print(used_forbidden)

# Check for product flag columns
product_flag_columns = [
    col for col in df.columns
    if "product" in col.lower() and "flag" in col.lower()
]

print("\nProduct-flag columns found in dataset:")
print(product_flag_columns)

# Final assertion
assert not used_forbidden, "Leakage detected!"

print("\nLeakage check: PASS")
print("The baseline score uses only the two declared observed signals.")

Features used by the score:
['days_since_last_update', 'impressions_90d']

Forbidden outcome/future fields used:
[]

Product-flag columns found in dataset:
[]

Leakage check: PASS
The baseline score uses only the two declared observed signals.


### Weak picks

The weakest picks are the rows near the bottom of the top-20 boundary. Their ranking may be sensitive to the relative staleness and visibility values.

A page can also be a weak practical recommendation if its content is still accurate despite being old, or if its search visibility does not represent meaningful business value.

### Leakage check

The baseline score uses only `days_since_last_update` and `impressions_90d`.

I did not use `trend_direction`, `trend_pct`, or `is_declining_label` to calculate the score. These fields are related to the outcome and are excluded from the scoring logic.

I also did not use future-window information or product flags. The result is a transparent decision-support baseline based on observed inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.